# Laboratorio 1 - Preparación de un corpus y EDA

Notebook paso a paso, con celdas de explicación y comentarios en el código.

**Corpus:** Spanish News Classification  

## 1. Objetivo

Explorar el corpus, aplicar el pipeline de normalización visto en clase y responder las preguntas de EDA y análisis.

In [21]:
# Importaciones
from collections import Counter
import re
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import spacy
import nltk
from nltk.corpus import stopwords as nltk_stopwords
from wordcloud import WordCloud

# Descargar recursos de NLTK (solo la primera vez)
nltk.download("stopwords")

# Cargar modelo de español
nlp = spacy.load("es_core_news_sm")

# Stopwords en español
ES_STOPWORDS = set(nltk_stopwords.words("spanish"))

# Rutas del proyecto
INPUT = Path("df_total.csv")          # Archivo de entrada
OUTDIR = Path("lab1_nlp_final")       # Carpeta de salida
OUTDIR.mkdir(exist_ok=True)

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/michelle/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [22]:
# Cargar el corpus
df = pd.read_csv(INPUT)
df.head()

,url,news,Type
0,https://www.larepublica.co/redirect/post/3201905,Durante el foro La banca articulador empresari...,Otra
1,https://www.larepublica.co/redirect/post/3210288,El regulador de valores de China dijo el domin...,Regulaciones
2,https://www.larepublica.co/redirect/post/3240676,En una industria históricamente masculina como...,Alianzas
3,https://www.larepublica.co/redirect/post/3342889,Con el dato de marzo el IPC interanual encaden...,Macroeconomia
4,https://www.larepublica.co/redirect/post/3427208,Ayer en Cartagena se dio inicio a la versión n...,Otra


## 2. Explorar el corpus antes de tocarlo

Preguntas: cuántos documentos hay, qué columnas contiene, cómo se distribuyen las categorías y si hay vacíos o duplicados.

In [23]:
# Resumen básico del corpus
print("Documentos:", len(df))
print("Columnas:", df.columns.tolist())
print("\nVacíos por columna:\n", df.isna().sum())
print("\nFilas duplicadas:", int(df.duplicated().sum()))
print("\nDistribución de categorías:\n", df["Type"].value_counts())

Documentos: 1217
Columnas: ['url', 'news', 'Type']

Vacíos por columna:
 url     0
news    0
Type    0
dtype: int64

Filas duplicadas: 75

Distribución de categorías:
 Type
Macroeconomia     340
Alianzas          247
Innovacion        195
Regulaciones      142
Sostenibilidad    137
Otra              130
Reputacion         26
Name: count, dtype: int64


### Respuestas de exploración

- El corpus tiene **1217 documentos**.
- Las columnas son **url**, **news** y **Type**.
- `url` guarda el enlace original, `news` guarda el texto completo y `Type` la categoría.
- Se detectaron **0 filas vacías** y **75 filas duplicadas**.
- La distribución de categorías aparece en la tabla siguiente.

In [24]:
# Distribución de categorías
category_counts = df["Type"].value_counts()
category_counts

Type
Macroeconomia     340
Alianzas          247
Innovacion        195
Regulaciones      142
Sostenibilidad    137
Otra              130
Reputacion         26
Name: count, dtype: int64

## 3. Preparación del corpus

Se aplica el pipeline en este orden: tokenización, minúsculas, eliminación de puntuación, eliminación de stopwords y lematización.

Nota: como no está instalado un modelo español completo de spaCy en este entorno, la última etapa se implementa como **stemming** con `SpanishStemmer` como aproximación.

In [25]:
# Configuración del preprocesamiento
punct_re = re.compile(r"\w+|[^\w\s]", flags=re.UNICODE)
word_re = re.compile(r"^\w+$", flags=re.UNICODE)
stopwords = set(ES_STOPWORDS) | {"rt", "http", "https", "www", "com"}

# Tokenizar: divide cada texto en palabras y signos
def tokenize(text):
    return punct_re.findall(str(text))

# Pasar a minúsculas
def to_lower(tokens):
    return [t.lower() for t in tokens]

# Quitar puntuación
def remove_punct(tokens):
    return [t for t in tokens if word_re.match(t)]

# Quitar stopwords
def remove_stop(tokens):
    return [t for t in tokens if t not in stopwords]

# Lematizar con spaCy
def lemmatize_tokens(tokens):
    doc = nlp(" ".join(tokens))
    return [token.lemma_ for token in doc if token.lemma_ != "-PRON-"]

# Normalización final
def normalize_tokens(text):
    tokens = tokenize(text)
    tokens = to_lower(tokens)
    tokens = remove_punct(tokens)
    tokens = remove_stop(tokens)
    tokens = lemmatize_tokens(tokens)
    return tokens

In [26]:
# Medir tokens y tipos después de cada paso
def flatten(series):
    return [t for doc in series for t in doc]

stage_tokens = {
    "Tokenización": flatten(df["news"].apply(tokenize)),
    "Minúsculas": flatten(df["news"].apply(lambda x: to_lower(tokenize(x)))),
    "Eliminar puntuación": flatten(df["news"].apply(lambda x: remove_punct(to_lower(tokenize(x))))),
    "Eliminar stopwords": flatten(df["news"].apply(lambda x: remove_stop(remove_punct(to_lower(tokenize(x)))))),
    "Lematización": flatten(df["news"].apply(normalize_tokens)),
}

stage_summary = pd.DataFrame([
    {"Paso": stage, "Tokens": len(tokens), "Tipos": len(set(tokens))}
    for stage, tokens in stage_tokens.items()
])

stage_summary["Reducción de vocabulario vs. tokenización (%)"] = (
    (stage_summary.loc[0, "Tipos"] - stage_summary["Tipos"]) / stage_summary.loc[0, "Tipos"] * 100
).round(2)

stage_summary

,Paso,Tokens,Tipos,Reducción de vocabulario vs. tokenización (%)
0,Tokenización,684297,32606,0.00
1,Minúsculas,684297,29571,9.31
2,Eliminar puntuación,637564,29530,9.43
3,Eliminar stopwords,339170,29315,10.09
4,Lematización,339206,22398,31.31


## 4. ¿Qué es un EDA?

EDA significa Análisis Exploratorio de Datos. Su objetivo es entender la estructura del corpus, detectar patrones y revisar calidad antes de modelar.

In [27]:
# Aplicar la normalización completa al corpus
df["tokens"] = df["news"].apply(normalize_tokens)
df["token_count"] = df["tokens"].str.len()
df.head()

,url,news,Type,tokens,token_count
0,https://www.larepublica.co/redirect/post/3201905,Durante el foro La banca articulador empresari...,Otra,"[foro, banca, articulador, empresarial, desarr...",110
1,https://www.larepublica.co/redirect/post/3210288,El regulador de valores de China dijo el domin...,Regulaciones,"[regulador, valor, chino, decir, domingo, busc...",186
2,https://www.larepublica.co/redirect/post/3240676,En una industria históricamente masculina como...,Alianzas,"[industria, históricamente, masculino, aviació...",201
3,https://www.larepublica.co/redirect/post/3342889,Con el dato de marzo el IPC interanual encaden...,Macroeconomia,"[dato, marzo, ipc, interanual, encadenar, deci...",259
4,https://www.larepublica.co/redirect/post/3427208,Ayer en Cartagena se dio inicio a la versión n...,Otra,"[ayer, cartagena, dar, inicio, versión, número...",420


In [28]:
# Estadísticas globales
all_norm_tokens = flatten(df["tokens"])
freq = Counter(all_norm_tokens)
global_tokens = len(all_norm_tokens)
global_types = len(set(all_norm_tokens))
global_ttr = global_types / global_tokens
reduction_total = (len(set(stage_tokens["Tokenización"])) - global_types) / len(set(stage_tokens["Tokenización"])) * 100

pd.DataFrame([
    ["Tokens totales normalizados", global_tokens],
    ["Tipos totales normalizados", global_types],
    ["Riqueza léxica global (type/token)", round(global_ttr, 4)],
    ["Reducción total del vocabulario (%)", round(reduction_total, 2)],
], columns=["Métrica", "Valor"])

,Métrica,Valor
0,Tokens totales normalizados,339206.000
1,Tipos totales normalizados,22398.000
2,Riqueza léxica global (type/token),0.066
3,Reducción total del vocabulario (%),31.310


In [29]:
# Tabla de frecuencias: 20 palabras más frecuentes
top20 = pd.DataFrame(freq.most_common(20), columns=["Palabra", "Frecuencia"])
top20

,Palabra,Frecuencia
0,poder,2531
1,bbva,2527
2,año,2240
3,banco,1561
4,país,1524
5,empresa,1410
6,inflación,1404
7,nuevo,1292
8,hacer,1257
9,precio,1247


In [30]:
# Riqueza léxica por categoría
cat_rows = []
for cat, g in df.groupby("Type"):
    toks = flatten(g["tokens"])
    cat_rows.append({
        "Categoria": cat,
        "Tokens": len(toks),
        "Tipos": len(set(toks)),
        "type_token_ratio": len(set(toks)) / len(toks),
    })
category_richness = pd.DataFrame(cat_rows).sort_values("type_token_ratio", ascending=False)
category_richness

,Categoria,Tokens,Tipos,type_token_ratio
5,Reputacion,7815,2537,0.324632
4,Regulaciones,37783,7026,0.185957
0,Alianzas,49927,8666,0.173573
3,Otra,29092,4974,0.170975
6,Sostenibilidad,57540,8052,0.139937
1,Innovacion,61183,6871,0.112302
2,Macroeconomia,95866,9421,0.098273
